# Tutorial 04 — Multi-Agent System: Replacement 01
## Building a CrewAI / LangGraph / AutoGen-like System with ONLY OpenAI SDK Primitives

**Part of the Agentic AI Tutorial Series** | `tutorials/02-agentic-ai/`

---

## 📋 Prerequisites

| Requirement | Details |
|---|---|
| **Conda Environment** | `agentic_ai` (Python 3.13.14) — activated via `conda activate agentic_ai` |
| **Key Packages** | `openai>=2.52.0`, `openai-agents>=0.19.2`, `python-dotenv`, `pydantic` |
| **API Key** | `DEEPSEEK_API_KEY` is already set in the `.env` file in this directory |
| **API Endpoint** | `https://api.deepseek.com/v1` (OpenAI-compatible ChatCompletion API) |
| **GPU** | Not required — all inference is API-based (DeepSeek cloud) |
| **Kernel** | Select the `agentic_ai` kernel in Jupyter before running |

> **✅ The `.env` file already exists** in `tutorials/02-agentic-ai/.env` with `DEEPSEEK_API_KEY` configured. No setup needed.

---

## 🎯 Learning Objectives

1. Build a **multi-agent collaboration system** using only OpenAI SDK primitives (`Agent`, `Runner`, `function_tool`, `handoff`)
2. Understand the **tree-structured knowledge topology** (`Structure` + `Node`) that enables 100× context-length amplification
3. Implement **role-based agent separation** with fine-grained tool permissions
4. Design a **cybernetic control loop** with state feedback for semi-automated orchestration
5. Grasp the **information-theoretic foundations** of agent capability boundaries

---

## 🧠 System Architecture at a Glance

```mermaid
flowchart TD
    U[👤 User Request] --> R[Representor<br/>Request Classifier]
    R -->|simple Q&A| A[Direct Answer]
    R -->|complex task| T[Thinker<br/>Ideation]
    T -->|create ideas| P[Planner<br/>Task Decomposition]
    P -->|plan tree| E1[Executor 1]
    P -->|plan tree| E2[Executor 2]
    P -->|plan tree| EN[Executor N...]
    E1 & E2 & EN -->|parallel results| RV[Reviewer<br/>Quality Gate]
    RV -->|incomplete + feedback| T
    RV -->|COMPLETE| I[Integrator<br/>Synthesis]
    I --> F[FinalAnswer<br/>Output Layer]
    F --> U2[👤 User Output]

    style R fill:#4a90d9,color:#fff
    style T fill:#50b86c,color:#fff
    style P fill:#e8a838,color:#fff
    style RV fill:#d94a4a,color:#fff
    style I fill:#9b59b6,color:#fff
```

**Core innovation**: A 128K context-window model coordinates **10M+ tokens** of effective computation through hierarchical compression and parallel execution.

## Introduction and Background

This tutorial presents a fully **OpenAI SDK-native multi-agent collaboration system** for open-ended planning and execution tasks. Its core innovation is a **Knowledge Infrastructure** built through tree-structured hierarchical compression, allowing a model with only 128K tokens of context to coordinate **10M+ tokens** of effective computation — roughly 100× beyond the model's nominal context length.

### Framework Comparison: Context Utilization

```mermaid
graph LR
    subgraph "Context Window: 128K tokens"
        A[CrewAI<br/>8-12K] 
        B[AutoGen<br/>5-20K]
        C[Raw OpenAI SDK<br/>10-15K]
        D[LangGraph<br/>70-90K]
        E[Replacement 01<br/>10M+ effective]
    end
    
    style A fill:#ff6b6b,color:#fff
    style B fill:#ff6b6b,color:#fff
    style C fill:#ffa502,color:#fff
    style D fill:#ffa502,color:#fff
    style E fill:#2ed573,color:#fff
```

| Framework | Effective Context | Bottleneck |
|---|---|---|
| **CrewAI** | 8K–12K tokens | Sequential task-output passing fills context |
| **AutoGen** | 5K–20K tokens | Dialogue message growth clogs the window |
| **Raw OpenAI SDK** | 10K–15K tokens | No built-in state management |
| **LangGraph** | 70K–90K tokens | Graph-based trimming, but still linear |
| **Replacement 01** | **10M+ effective** | Hierarchical compression + parallel execution |

### Design Philosophy

Replacement 01 borrows from three major paradigms:
- **CrewAI**: Role-based multi-agent definitions with clear responsibility boundaries
- **AutoGen**: Inter-agent message exchange patterns
- **LangGraph**: The concept of state machines, but embedded into the **data topology itself** rather than an explicit graph API

The result: using only `Agent`, `Runner`, `function_tool`, and `handoff` from the OpenAI Agents SDK, we build a system whose **conceptual complexity exceeds heavyweight frameworks** while maintaining **orders-of-magnitude better token efficiency**.

## 1. Data Structures — The Knowledge Topology

Unlike LangGraph's explicit `StateGraph` or CrewAI's implicit task-output passing, Replacement 01 builds an **explicit, tree-shaped, self-describing knowledge structure**: the `Structure` and `Node` classes.

### 1.1 Tree Topology Visualization

```mermaid
graph TD
    S[Structure] --> I0[Idea 0<br/>Wild Quotient]
    S --> I1[Idea 1<br/>Weighted Hypersurface]
    S --> I2[Idea N...]
    
    I0 --> P0[Plan 0: Choose Base]
    I0 --> P1[Plan 1: Design Action]
    I0 --> P2[Plan 2: Analyze]
    
    P0 --> S0[Step 0.0<br/>Consider P²]
    P0 --> S1[Step 0.1<br/>Use deg-1 dP]
    P0 --> S2[Step 0.2<br/>Analyze Aut]
    
    S0 --> L0[Leaf: P² coords<br/>results: 100 tok]
    S1 --> L1[Leaf: deg-1<br/>results: 120 tok]
    S2 --> L2[Leaf: PGL₃<br/>results: 150 tok]

    style S fill:#4a90d9,color:#fff
    style I0 fill:#50b86c,color:#fff
    style I1 fill:#50b86c,color:#fff
    style I2 fill:#50b86c,color:#fff
    style L0 fill:#ffa502,color:#fff
    style L1 fill:#ffa502,color:#fff
    style L2 fill:#ffa502,color:#fff
```

### 1.2 Node Fields

| Field | Type | Purpose |
|---|---|---|
| `name` | `str` | Human-readable task summary |
| `description` | `str` | Detailed task context for agents |
| `status` | `Literal["pending","in_progress","completed","blocked"]` | State-machine marker |
| `results` | `Optional[str]` | **Compressed conclusions** (the key to amplification) |
| `children` | `List[Node]` | Sub-tasks (max 6 — bounded branching) |
| `session` | `List[str]` | Debug logs (NOT injected into agent context) |

### 1.3 Node State Machine

```mermaid
stateDiagram-v2
    [*] --> pending: Node created
    pending --> in_progress: Executor picks up
    in_progress --> completed: Task succeeds
    in_progress --> blocked: Task fails
    blocked --> pending: Replan (Thinker)
    completed --> [*]
```

### 1.4 Why This Beats Message Lists

Traditional frameworks model agent communication as **linear message sequences** that grow with every interaction round. These contain redundant information: repeated system prompts, raw JSON tool calls, uncompressed intermediate outputs. Once the context window fills with low-value data, **effective attention collapses**.

Our design is **knowledge-centered**: the message list is demoted to a temporary, stateless medium, while `Structure` handles all knowledge accumulation. This **fundamentally solves context decay**.

## 2. Multi-Agent System — Roles & Tool Permissions

### 2.1 Agent Role Map

```mermaid
graph TD
    subgraph "Entry Layer"
        REP[Representor<br/>🛡️ Request Classifier<br/>Tools: handoff only]
    end
    
    subgraph "Planning Layer"
        THK[Thinker<br/>💡 Ideation<br/>Tools: list_ideas, create_idea_node, execute_project_plan]
        PLN[Planner<br/>📋 Decomposition<br/>Tools: get_structure_summary, create_plan_node, finish_planning]
    end
    
    subgraph "Execution Layer"
        EX1[Executor 1<br/>⚙️ Bottom-up leaf processing]
        EX2[Executor 2<br/>⚙️]
        EXN[Executor N<br/>⚙️]
    end
    
    subgraph "Quality Layer"
        REV[Reviewer<br/>🔍 Quality Gate]
    end
    
    subgraph "Output Layer"
        INT[Integrator<br/>🧩 Synthesis]
        FIN[FinalAnswer<br/>📤 Output]
    end
    
    REP -->|complex| THK
    THK --> PLN
    PLN --> EX1 & EX2 & EXN
    EX1 & EX2 & EXN --> REV
    REV -->|incomplete| THK
    REV -->|COMPLETE| INT
    INT --> FIN
    
    style REP fill:#4a90d9,color:#fff
    style THK fill:#50b86c,color:#fff
    style PLN fill:#e8a838,color:#fff
    style REV fill:#d94a4a,color:#fff
    style INT fill:#9b59b6,color:#fff
    style FIN fill:#2c3e50,color:#fff
```

### 2.2 Least-Privilege Tool Assignment

| Agent | Tools | Why Limited? |
|---|---|---|
| **Representor** | None (handoff only) | Prevents launching heavy workflows for simple Q&A |
| **Thinker** | `list_ideas`, `create_idea_node`, `execute_project_plan` | Only ideation + trigger — no execution |
| **Planner** | `get_structure_summary`, `create_plan_node`, `finish_planning` | Only tree construction — no status changes |
| **Executor** | `get_structure_summary`, `update_node_status`, `log_to_node` | Only leaf processing — no tree restructuring |
| **Reviewer** | `get_structure_summary`, `set_feedback` | Only observation + feedback — no mutation |
| **Integrator** | `get_structure_summary` | Read-only synthesis — no writes |
| **FinalAnswer** | None | Pure output passthrough |

### 2.3 Token Amplification Math

If a 4-layer tree is fully expanded (6 children per node):

| Layer | Nodes | Tokens per node | Total |
|---|---|---|---|
| Idea | 6 | — | — |
| Plan | 36 | — | — |
| Step | 216 | — | — |
| **Leaf** | **1,296** | ~3,200 (1K in + 200 out + 2K reasoning) | **~4.15M** |
| Planner | — | — | ~150K |
| **Total effective** | — | — | **~4.3M tokens** |

Every single model call stays under 5K tokens. The amplification comes from **hierarchical compression** via `results` fields.

## 3. Control Loop — The Cybernetic Heart

```mermaid
sequenceDiagram
    participant U as 👤 User
    participant R as Representor
    participant T as Thinker
    participant P as Planner
    participant E as Executor(s)
    participant RV as Reviewer
    participant I as Integrator
    participant F as FinalAnswer

    U->>R: Complex request
    R->>T: Handoff
    T->>T: create_idea_node × 6
    T->>T: execute_project_plan
    
    loop Control Loop (max 3)
        T->>P: Expand ideas
        P->>P: create_plan_node × N
        P->>P: finish_planning
        
        par Parallel Execution
            T->>E: Process idea 0
            T->>E: Process idea 1
            T->>E: Process idea N
        end
        
        E-->>RV: Results ready
        RV->>RV: get_structure_summary
        
        alt All completed
            RV-->>I: COMPLETE
            I->>F: Synthesized answer
            F-->>U: Final output
        else Blocked/Pending
            RV->>T: set_feedback
            Note over T: Replan with feedback
        end
    end
```

### Cybernetic Principle

> *"If an unstable system is fully controllable, it can be stabilized through state feedback."*

- LLM outputs are **stochastic** → the system is inherently "unstable"
- The `Reviewer` acts as a **state observer**
- Feedback is injected into `Thinker` as **control input correction**
- `max_loops` provides the **deterministic safety bound**

### Semi-Automation Design

| Decision | Made By |
|---|---|
| Create new ideas? | `Thinker` (agent) |
| How to refine plans? | `Planner` (agent) |
| Execution order? | `Executor` bottom-up search (agent) |
| Is task complete? | `Reviewer` (agent) |
| Loop iteration count | Python code (deterministic) |
| Max turns per call | Python code (deterministic) |

This gives **program-level determinism** with **agent-level flexibility**.

## 4. Information-Theoretic Foundation

### LLM as an Information Channel

```mermaid
graph LR
    P[Prompt<br/>Context Window] -->|Channel Capacity C| LLM[LLM<br/>Noisy Channel]
    LLM -->|Generated Text| O[Output]
    
    subgraph "Constraints"
        C1[Model Scale]
        C2[Training Quality]
        C3[Context Length]
    end
    
    C1 --> LLM
    C2 --> LLM
    C3 --> LLM
```

**Data Processing Inequality**: In a cascade of channels, total capacity ≤ min(individual capacities).

> If the base model lacks instruction-following ability, **no agent framework can compensate**.

### Why DeepSeek-V3?

This implementation uses **DeepSeek-V3** (`deepseek-chat`) as the uniform backend through the standard **ChatCompletion API** (`POST /v1/chat/completions`) because:

| Capability | Why It Matters for Agents |
|---|---|
| **Instruction following** | Must strictly obey tool-choice instructions without filler text |
| **Structured output** | Must reliably produce `results` in the expected format |
| **Long-context attention** | Must extract information accurately from thousands of tokens |
| **Low cost** | ~$0.27/M input tokens — enables 10M+ token experiments affordably |

No "freedom configuration" or special fine-tuning is used — the system relies purely on architectural innovation within the standard ChatCompletion API.

---

## 🔧 Cell 1: Setup — Imports & API Client

This cell sets up:
- All imports (async, Pydantic, `openai-agents` SDK)
- The `.env` file loader — reads the pre-configured `DEEPSEEK_API_KEY`
- The `AsyncOpenAI` client pointed at DeepSeek's **ChatCompletion API** (`https://api.deepseek.com/v1`)
- The `OpenAIChatCompletionsModel` wrapper that bridges the Agents SDK to the ChatCompletion endpoint

> **Note:** `.env` already exists at `tutorials/02-agentic-ai/.env` with your API key.

In [1]:
# ============================================================================
# CELL 1: IMPORTS & API CLIENT SETUP
# ============================================================================
import asyncio
import os
import json
import traceback
from datetime import datetime
from dataclasses import dataclass
from typing import List, Optional, Literal

from dotenv import load_dotenv          # Reads DEEPSEEK_API_KEY from .env
from openai import AsyncOpenAI           # Async client for ChatCompletion API
from pydantic import BaseModel, Field    # Data validation & serialization

from agents import (
    Agent, Runner, SQLiteSession, function_tool, handoff,
    OpenAIChatCompletionsModel, set_tracing_disabled, RunContextWrapper
)
# `agents` = openai-agents package. We compose Agent/Runner primitives
# into a full multi-agent system — no LangGraph/CrewAI/AutoGen needed.

# --------------------------------------------------------------------------
# Load .env and configure DeepSeek ChatCompletion API
# --------------------------------------------------------------------------
load_dotenv(override=True)  # Loads DEEPSEEK_API_KEY from .env in this directory
set_tracing_disabled(True)   # Disable OpenAI tracing to reduce overhead

# Create async client pointed at DeepSeek's OpenAI-compatible endpoint
# Uses the standard ChatCompletion API: POST /v1/chat/completions
client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

# Wrap in OpenAIChatCompletionsModel for the Agents SDK
# Uses deepseek-chat (DeepSeek-V3) for all agents — no reasoning model needed
model = OpenAIChatCompletionsModel(model="deepseek-chat", openai_client=client)

print("✅ Setup complete — DeepSeek ChatCompletion API configured")
print(f"   Model: deepseek-chat | Endpoint: https://api.deepseek.com/v1")
print(f"   API Key loaded: {'Yes' if os.getenv('DEEPSEEK_API_KEY') else 'No — check .env file!'}")

✅ Setup complete — DeepSeek ChatCompletion API configured
   Model: deepseek-chat | Endpoint: https://api.deepseek.com/v1
   API Key loaded: Yes


## 🔧 Cell 2: Debug Logging & Data Structures

This cell defines the **skeleton** of the entire system:
- **Logging utilities**: `log_event`, `log_tool_call`, `log_structure` — real-time visibility into agent workflow
- **`Node`**: The fundamental unit of the knowledge tree — a Pydantic model with a built-in state machine
- **`Structure`**: Top-level container for 6 ideas + Reviewer feedback
- **`ProjectContext`**: Shared context passed to all agents via `RunContextWrapper`

```mermaid
classDiagram
    class Node {
        +str name
        +str description
        +Optional~str~ results
        +Literal status
        +List~Node~ children
        +List~str~ session
        +add_child(Node) void
    }
    class Structure {
        +List~Node~ ideas
        +Optional~str~ feedback
        +add_idea(Node) void
        +get_node_by_path(List~int~) Optional~Node~
    }
    class ProjectContext {
        +Structure structure
        +Optional~str~ final_answer
        +Optional~SQLiteSession~ session
    }
    Structure --> Node : contains up to 6
    ProjectContext --> Structure : wraps
```

In [2]:
# ============================================================================
# CELL 2: DEBUG LOGGING UTILITIES & DATA STRUCTURES
# ============================================================================

# ---- Debug Logging ----
def log_event(agent_name: str, event: str, details: str = ""):
    """Log an agent lifecycle event with timestamp."""
    timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"[{timestamp}] 🟢 {agent_name:15} | {event:25} | {details}")

def log_tool_call(agent_name: str, tool_name: str, args: dict):
    """Log a tool invocation with truncated arguments."""
    args_str = json.dumps(args, default=str)[:100]
    print(f"[{datetime.now().strftime('%H:%M:%S.%f')[:-3]}] 🔧 {agent_name:15} | TOOL: {tool_name:20} | {args_str}")

def log_structure(structure: "Structure"):
    """Pretty-print the knowledge tree with statuses."""
    print("\n" + "="*60)
    print("📊 CURRENT STRUCTURE")
    print("="*60)
    if not structure.ideas:
        print("(No ideas yet)")
    else:
        for i, idea in enumerate(structure.ideas):
            print(f"🌱 Idea {i}: {idea.name} [{idea.status}]")
            for j, plan in enumerate(idea.children):
                print(f"   📋 Plan {i}.{j}: {plan.name} [{plan.status}]")
                for k, step in enumerate(plan.children):
                    print(f"      🔧 Step {i}.{j}.{k}: {step.name} [{step.status}]")
                    if step.results:
                        print(f"         → Result: {step.results[:50]}...")
    print("="*60 + "\n")


# ---- Data Structures: The Knowledge Topology ----
class Node(BaseModel):
    """
    A single node in the knowledge tree. Each Node is an information container.
    The `results` field stores COMPRESSED output — this is the core mechanism
    that enables 100×+ context-length amplification.
    """
    name: str
    description: str = ""
    results: Optional[str] = None           # Compressed output (100-200 tokens)
    status: Literal["pending", "in_progress", "completed", "blocked"] = "pending"
    children: List["Node"] = Field(default_factory=list)   # Max 6 children
    session: List[str] = Field(default_factory=list)        # Debug log (not in context)

    def add_child(self, child: "Node"):
        """Enforce max 6 children (bounded branching factor)."""
        if len(self.children) >= 6:
            raise ValueError("Maximum 6 children per node")
        self.children.append(child)

Node.model_rebuild()  # Required for Pydantic forward references in recursive types


@dataclass
class ProjectContext:
    """
    Shared context passed to all agents via RunContextWrapper.
    This is the CENTRAL STATE that all agents read and mutate through tools.
    """
    structure: "Structure"
    final_answer: Optional[str] = None
    session: Optional[SQLiteSession] = None


class Structure(BaseModel):
    """
    Top-level container for the knowledge tree.
    - Holds up to 6 idea nodes (bounded branching at root)
    - Stores Reviewer feedback for the control loop
    - Serializable to JSON via model_dump_json() for cross-session persistence
    """
    ideas: List[Node] = Field(default_factory=list)
    feedback: Optional[str] = None

    def add_idea(self, idea: Node):
        """Add a top-level idea (max 6)."""
        if len(self.ideas) >= 6:
            raise ValueError("Maximum 6 ideas")
        self.ideas.append(idea)

    def get_node_by_path(self, path: List[int]) -> Optional[Node]:
        """
        Navigate tree by index path.
        Example: path=[0, 2, 1] → ideas[0].children[2].children[1]
        """
        if not path:
            return None
        current = self.ideas[path[0]]
        for idx in path[1:]:
            if idx >= len(current.children):
                return None
            current = current.children[idx]
        return current

print("✅ Data structures defined: Node, Structure, ProjectContext")

✅ Data structures defined: Node, Structure, ProjectContext


## 🔧 Cell 3: Shared Tool Definitions

These `@function_tool`-decorated functions are the **ONLY way agents interact** with the knowledge tree. Each agent gets a subset based on its role.

### Tool-to-Agent Mapping

```mermaid
graph LR
    T1[list_ideas] --> THK[Thinker]
    T2[create_idea_node] --> THK
    T6[get_structure_summary] --> THK
    T9[execute_project_plan] --> THK

    T3[create_plan_node] --> PLN[Planner]
    T6 --> PLN
    T8[finish_planning] --> PLN

    T4[update_node_status] --> EX[Executor]
    T5[log_to_node] --> EX
    T6 --> EX

    T6 --> RV[Reviewer]
    T7[set_feedback] --> RV

    T6 --> INT[Integrator]
```

`get_structure_summary` is shared by **5 of 6 agents** — it's the backbone of context-length amplification. Thinker needs it when replanning after Reviewer feedback.

In [3]:
# ============================================================================
# CELL 3: SHARED TOOL DEFINITIONS
# ============================================================================
# These are the ONLY functions agents use to interact with the knowledge tree.
# Each agent gets a SUBSET based on its role (least-privilege principle).

@function_tool
async def list_ideas(wrapper: RunContextWrapper[ProjectContext]) -> str:
    """List all top-level ideas with statuses. → Thinker"""
    ideas = wrapper.context.structure.ideas
    if not ideas:
        return "No ideas yet."
    lines = [f"{i}: {idea.name} [{idea.status}]" for i, idea in enumerate(ideas)]
    log_tool_call("(Tool)", "list_ideas", {})
    return "\n".join(lines)


@function_tool
async def create_idea_node(wrapper: RunContextWrapper[ProjectContext], name: str, description: str) -> str:
    """Create a new top-level idea (max 6). → Thinker"""
    idea = Node(name=name, description=description)
    wrapper.context.structure.add_idea(idea)
    log_tool_call("Thinker", "create_idea_node", {"name": name})
    return f"Idea '{name}' created."


@function_tool
async def create_plan_node(
    wrapper: RunContextWrapper[ProjectContext], parent_path: List[int],
    name: str, description: str
) -> str:
    """
    Create a child node under parent_path. → Planner
    parent_path is a list of indices: [0, 1] = ideas[0].children[1]
    """
    parent = wrapper.context.structure.get_node_by_path(parent_path)
    if not parent:
        return f"Error: node at path {parent_path} not found."
    child = Node(name=name, description=description)
    parent.add_child(child)
    log_tool_call("Planner", "create_plan_node", {"parent": parent_path, "name": name})
    return f"Node '{name}' added under '{parent.name}'."


@function_tool
async def update_node_status(
    wrapper: RunContextWrapper[ProjectContext], path: List[int],
    status: Literal["pending", "in_progress", "completed", "blocked"],
    results: Optional[str] = None, description_update: Optional[str] = None,
) -> str:
    """
    Update node status + optionally results/description. → Executor
    This is the PRIMARY state-transition mechanism in the knowledge tree.
    """
    node = wrapper.context.structure.get_node_by_path(path)
    if not node:
        return f"Node at path {path} not found."
    node.status = status
    if results is not None:
        node.results = results
    if description_update is not None:
        node.description = description_update
    log_tool_call("Executor", "update_node_status", {"path": path, "status": status})
    return f"Node '{node.name}' updated."


@function_tool
async def log_to_node(wrapper: RunContextWrapper[ProjectContext], path: List[int], message: str) -> str:
    """
    Append debug message to node's session log. → Executor
    NOTE: Session logs are NOT injected into planning/integration context,
    avoiding token pollution while maintaining traceability.
    """
    node = wrapper.context.structure.get_node_by_path(path)
    if not node:
        return f"Node at path {path} not found."
    node.session.append(message)
    log_tool_call("Executor", "log_to_node", {"path": path, "message": message[:30]})
    return f"Logged to node '{node.name}'."


@function_tool
async def get_structure_summary(wrapper: RunContextWrapper[ProjectContext]) -> str:
    """
    Return a CONCISE summary of the entire knowledge tree.
    This is the KEY tool for context-length amplification — agents see
    only status summaries, not full execution traces.
    → Planner, Executor, Reviewer, Integrator
    """
    lines = []
    for i, idea in enumerate(wrapper.context.structure.ideas):
        lines.append(f"Idea {i}: {idea.name} [{idea.status}] - {len(idea.children)} plans")
        for j, plan in enumerate(idea.children):
            lines.append(f"  Plan {j}: {plan.name} [{plan.status}] - {len(plan.children)} steps")
    summary = "\n".join(lines) if lines else "Structure is empty."
    log_tool_call("(Tool)", "get_structure_summary", {})
    print(f"[DEBUG] get_structure_summary returned:\n{summary}\n---")
    return summary


@function_tool
async def set_feedback(wrapper: RunContextWrapper[ProjectContext], feedback: str) -> str:
    """
    Store Reviewer feedback for the control loop. → Reviewer
    This feedback drives the cybernetic state-feedback mechanism:
    Thinker reads it during replanning to correct system behavior.
    """
    wrapper.context.structure.feedback = feedback
    log_tool_call("Reviewer", "set_feedback", {"feedback": feedback[:50]})
    return "Feedback stored."


@function_tool
async def finish_planning(wrapper: RunContextWrapper[ProjectContext]) -> str:
    """Signal Planner completion to prevent free-form text output. → Planner"""
    log_tool_call("Planner", "finish_planning", {})
    return "Planning finished."


print("✅ 8 shared tools defined: list_ideas, create_idea_node, create_plan_node,")
print("   update_node_status, log_to_node, get_structure_summary, set_feedback, finish_planning")

✅ 8 shared tools defined: list_ideas, create_idea_node, create_plan_node,
   update_node_status, log_to_node, get_structure_summary, set_feedback, finish_planning


## 🔧 Cell 4: Sub-Agent Definitions

The **inner loop agents** — Planner, Executor, Reviewer, Integrator. Each has:
- **Specific role** with tightly scoped natural-language instructions
- **Limited toolset** following least-privilege
- **Same model**: `deepseek-chat` via the ChatCompletion API wrapper

Thinker, Representor, and FinalAnswer are defined later — they handle the outer flow.

In [4]:
# ============================================================================
# CELL 4: SUB-AGENT DEFINITIONS (Inner Loop)
# ============================================================================
# These four agents form the inner orchestration loop.
# Each has a narrow role and limited tools (least-privilege).

planner = Agent[ProjectContext](
    name="Planner",
    instructions="""
    You are a planner. Your ONLY job is to create a detailed plan tree.
    Follow these steps EXACTLY:
    1. Use `get_structure_summary` to see the current idea tree.
    2. Create at least 3 plan nodes under the target idea using `create_plan_node`.
    3. For each plan, create at least 2 sub-steps (second level).
    4. Optionally create third-level steps.
    5. Call `finish_planning` exactly once when done.
    Do NOT output any text. Only use the tools.
    """,
    tools=[get_structure_summary, create_plan_node, finish_planning],
    model=model,
)

executor = Agent[ProjectContext](
    name="Executor",
    instructions="""
    Work bottom-up on the assigned idea tree.
    - Use `get_structure_summary` to see the tree.
    - Find leaf nodes with status 'pending'.
    - For each: produce a short result, call `update_node_status(status='completed', results=...)`.
    - Mark blocked nodes with reason.
    Continue until all leaf nodes are processed, then stop.
    """,
    tools=[get_structure_summary, update_node_status, log_to_node],
    model=model,
)

reviewer = Agent[ProjectContext](
    name="Reviewer",
    instructions="""
    Examine the structure using `get_structure_summary`.
    - If any node is 'blocked' or 'pending', use `set_feedback` to explain.
    - If all nodes are 'completed', output "COMPLETE" (exact word, nothing else).
    """,
    tools=[get_structure_summary, set_feedback],
    model=model,
)

integrator = Agent[ProjectContext](
    name="Integrator",
    instructions="""
    Synthesize results from all idea trees into a final answer.
    - Use `get_structure_summary` to read all results.
    - Combine the best parts into a clear, comprehensive response.
    """,
    tools=[get_structure_summary],
    model=model,
)

print("✅ 4 sub-agents defined: Planner, Executor, Reviewer, Integrator")

✅ 4 sub-agents defined: Planner, Executor, Reviewer, Integrator


## 🔧 Cell 5: Orchestration Loop & Main Agents

This is the **heart** of the system — two components:

### `run_project_manager()` — The Control Loop

```mermaid
flowchart TD
    START([start]) --> LOOP{loop_idx < max_loops?}
    LOOP -->|yes| PLAN[Phase 1: Planning<br/>Planner expands ideas]
    PLAN --> EXEC[Phase 2: Parallel Execution<br/>asyncio.gather all Executors]
    EXEC --> REVIEW[Phase 3: Review<br/>Reviewer checks status]
    REVIEW --> CHECK{All completed?}
    CHECK -->|COMPLETE| INTEG[Phase 4: Integration<br/>Integrator synthesizes]
    CHECK -->|incomplete| FEEDBACK[Feedback → Thinker replans]
    FEEDBACK --> LOOP
    INTEG --> DONE([return final_answer])
    LOOP -->|no / max reached| FORCE[Force integration<br/>with partial results]
    FORCE --> DONE
```

### Main Agents & Handoffs

- **Representor** → routes simple vs complex requests (handoff to Thinker for complex)
- **Thinker** → generates up to 6 ideas, triggers `execute_project_plan`, then hands off to FinalAnswer
- **FinalAnswer** → pure output passthrough, isolates final answer from internal state

Handoff chain: `Representor → Thinker → FinalAnswer`

In [5]:
# ============================================================================
# CELL 5: ORCHESTRATION LOOP & MAIN AGENTS
# ============================================================================

async def run_project_manager(context: ProjectContext, max_loops: int = 3) -> str:
    """Cybernetic control loop: Planner → Executors(parallel) → Reviewer → Integrator."""
    session = context.session
    for loop_idx in range(max_loops):
        log_event("Orchestrator", f"Loop {loop_idx+1}/{max_loops}", "")
        log_structure(context.structure)

        # Phase 1: Planning
        for i, idea in enumerate(context.structure.ideas):
            if not idea.children:
                log_event("Orchestrator", "Calling Planner", f"Idea {i}: {idea.name}")
                try:
                    await Runner.run(planner,
                        f"Expand idea at [{i}] '{idea.name}'.",
                        context=context, session=session, max_turns=2000)
                    log_event("Orchestrator", "Planner done", f"{len(idea.children)} children")
                except Exception as e:
                    log_event("Orchestrator", "Planner ERROR", str(e))

        # Phase 2: Parallel Execution
        tasks = [Runner.run(executor, f"Process idea [{i}].",
                            context=context, session=session, max_turns=200)
                 for i in range(len(context.structure.ideas))]
        await asyncio.gather(*tasks)
        log_event("Orchestrator", "Executors finished", "")

        # Phase 3: Review
        review = await Runner.run(reviewer,
            "Review. Output 'COMPLETE' if all done, else set_feedback.",
            context=context, session=session)
        review_text = review.final_output.strip()
        log_event("Reviewer", "Output", review_text[:100])

        # Phase 4: Decision
        if review_text == "COMPLETE":
            int_result = await Runner.run(integrator,
                "Integrate all results.", context=context, session=session)
            context.final_answer = int_result.final_output
            return int_result.final_output
        else:
            # Cybernetic feedback: Reviewer → Thinker replanning
            # Thinker needs get_structure_summary to see current state
            await Runner.run(thinker,
                f"Feedback: {context.structure.feedback}. Adjust ideas.",
                context=context, session=session, max_turns=50000)

    # Fallback: max loops reached
    int_result = await Runner.run(integrator,
        "Integrate partial results.", context=context, session=session)
    context.final_answer = int_result.final_output
    return int_result.final_output


@function_tool
async def execute_project_plan(wrapper: RunContextWrapper[ProjectContext]) -> str:
    """Thinker's entry point to launch the orchestration loop."""
    log_tool_call("Thinker", "execute_project_plan", {})
    try:
        wrapper.context.final_answer = await run_project_manager(wrapper.context)
        return "Project execution completed."
    except Exception as e:
        log_event("Orchestrator", "FATAL", str(e))
        return f"Error: {e}"

# ---- Main Agents ----
representor = Agent[ProjectContext](
    name="Representor",
    instructions="Handoff to Thinker for complex requests. Otherwise answer directly.",
    tools=[], model=model,
)
thinker = Agent[ProjectContext](
    name="Thinker",
    instructions="1. list_ideas 2. create_idea_node (up to 6) 3. execute_project_plan 4. handoff to FinalAnswer.",
    tools=[list_ideas, create_idea_node, get_structure_summary, execute_project_plan],  # FIXED: added get_structure_summary for replanning
    model=model,
)
final_agent = Agent[ProjectContext](
    name="FinalAnswer",
    instructions="Output final_answer field exactly as stored.",
    tools=[], model=model,
)

representor.handoffs = [handoff(agent=thinker)]
thinker.handoffs = [handoff(agent=final_agent)]

log_event("Setup", "Ready", "Representor→Thinker→FinalAnswer")
print("✅ Full system ready: 7 agents, 9 tools, control loop configured")

[16:23:00.359] 🟢 Setup           | Ready                     | Representor→Thinker→FinalAnswer
✅ Full system ready: 7 agents, 9 tools, control loop configured


---

## 🧪 Cell 6: Run the Multi-Agent System

This cell initializes the knowledge infrastructure and launches the full agent chain.

### Test Prompts Available

| Prompt | Description | Approx. Tokens | Time |
|---|---|---|---|
| **Garden Planning** ✅ (active) | "I want to start a small vegetable garden..." | ~500K–1M | ~2–5 min |
| **Math Problem** (commented out) | KLT del Pezzo surface — unsolved research problem | ~10M–20M | ~15–30 min |

> **⚠️ The math problem is expensive in API credits!** Use the garden prompt for testing.
> The `agent_session.db` SQLite file persists all conversation history for session recovery.

### What Happens When You Run This

```mermaid
sequenceDiagram
    participant Cell as 🧪 This Cell
    participant R as Representor
    participant T as Thinker
    participant Loop as Control Loop
    participant User as 👤 You
    
    Cell->>R: Runner.run(representor, prompt)
    R->>T: Handoff (complex request detected)
    T->>T: create_idea_node × N ideas
    T->>Loop: execute_project_plan
    Loop->>Loop: Planner → Executors(parallel) → Reviewer
    Note over Loop: May loop up to 3 times<br/>if Reviewer finds issues
    Loop-->>Cell: Final integrated answer
    Cell-->>User: Display answer + structure JSON
```

In [6]:
# ============================================================================
# CELL 6: RUN THE MULTI-AGENT SYSTEM
# ============================================================================
# This is where the magic happens. The full agent chain processes your request:
# Representor → Thinker → Planner → Executors(parallel) → Reviewer → Integrator

# ---- Choose your test prompt ----
# Option A: Simple garden planning (quick test, ~2-5 min, ~500K-1M tokens)
user_prompt = (
    "I want to start a small vegetable garden in my backyard. "
    "I'm a complete beginner with no experience. Help me plan everything "
    "from site selection to harvest. Be very detailed and practical."
)

# Option B: Advanced math research problem (stress test, ~15-30 min, ~10M+ tokens)
# Uncomment the block below to use instead:
# user_prompt = """Solve the following unsolved math problem:
# Construct an explicit normal projective surface X over an algebraically closed
# field of characteristic 3 such that:
# (1) X is a klt del Pezzo surface
# (2) ρ(X) = 1 (Picard number 1)
# (3) X has more than seven singular points.
# You should actually solve it and provide verifiable proof. Try different
# approaches and split into many different steps."""

# ---- Initialize knowledge infrastructure ----
structure = Structure()                              # Empty knowledge tree
session = SQLiteSession("agent_session.db")          # Persistent conversation DB
context = ProjectContext(structure=structure, session=session)

log_event("System", "Starting workflow", f"Prompt: {user_prompt[:60]}...")
print("\n" + "🚀"*30)
print("AGENTIC AI MULTI-AGENT SYSTEM")
print("🚀"*30 + "\n")

# ---- Launch the agent chain ----
# Runner.run starts Representor → detects complexity → hands off to Thinker → ...
# max_turns=5000 allows deep trees with many tool calls and feedback loops.
result = await Runner.run(
    representor,
    user_prompt,
    context=context,
    session=session,
    max_turns=5000,
)

# ---- Display results ----
print("\n" + "🏁"*30)
print("FINAL OUTPUT")
print("🏁"*30)
print(result.final_output)

print("\n" + "📊"*30)
print("FINAL STRUCTURE (JSON) — serializable for cross-session recovery")
print("📊"*30)
print(context.structure.model_dump_json(indent=2))

[16:23:00.364] 🟢 System          | Starting workflow         | Prompt: I want to start a small vegetable garden in my backyard. I'm...

🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
AGENTIC AI MULTI-AGENT SYSTEM
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀

[16:23:07.059] 🔧 Thinker         | TOOL: create_idea_node     | {"name": "Site Selection & Preparation"}
[16:23:07.059] 🔧 Thinker         | TOOL: create_idea_node     | {"name": "Garden Design & Layout"}
[16:23:07.059] 🔧 Thinker         | TOOL: create_idea_node     | {"name": "Soil Building & Amendment"}
[16:23:07.059] 🔧 Thinker         | TOOL: create_idea_node     | {"name": "Plant Selection & Timing"}
[16:23:07.059] 🔧 Thinker         | TOOL: create_idea_node     | {"name": "Ongoing Care & Maintenance"}
[16:23:07.059] 🔧 Thinker         | TOOL: create_idea_node     | {"name": "Harvest & Post-Harvest"}
[16:23:08.094] 🔧 Thinker         | TOOL: execute_project_plan | {}
[16:23:08.094] 🟢 Orchestrator    | Loop 1/3                  | 

📊 CURRENT STRUCTURE
🌱 Idea 0: 

## 7. Notes and Limitations

It is important to emphasize that **Replacement 01** is a concept-validation toy model. For transparency:

- **No formal tuning**: The system relies entirely on the model's default output, without additional entropy control or decoding strategies.
- **No reasoning model**: All agents use standard `deepseek-chat` via the ChatCompletion API, not a reasoning-enhanced variant.
- **Simplified architecture**: Batch integration and cross-session recovery are not fully implemented. The token amplification figures are **theoretical estimates** based on architecture potential, not measured runtime data.
- **Positioning**: This is a **design paradigm proposal** and directional validation, not a complete industrial implementation.

---

### Contact

For job opportunities or project collaboration: `yucongcai_business@outlook.com`  
For research-related matters: `yucongcai_research@outlook.com`

---

## Version Log

| Version | Date | Change |
|---|---|---|
| v1.0 | 2026-08-03 | Initial rebuild from `assets/previous-resources/`. Fixed inverted child/idea limit logic. |
| v1.1 | 2026-08-04 | Documentation overhaul: added Python comments, prerequisites, architecture diagram. |
| **v1.2** | **2026-08-04** | **Major restructuring**: split monolithic code cell into 5 logical cells with markdown explanations. Added 7 Mermaid diagrams (architecture, framework comparison, tree topology, node state machine, agent role map, control sequence, orchestration flowchart, class diagram, tool-agent mapping, run sequence). Updated `.env` references to pre-configured file. Default test prompt set to garden planning (runnable out of the box). ChatCompletion API notes added throughout. |

### v1.2 Changes (2026-08-04)

| Change |
|---|
| Split 600-line monolithic code cell into 5 focused cells: Imports+Setup, Data Structures, Tools, Sub-Agents, Orchestration+Main Agents |
| Added markdown explanation before every code cell with purpose, role, and Mermaid diagrams |
| Added 10 Mermaid diagrams: architecture, framework comparison, tree topology, node state machine, agent role map, class diagram, control sequence, orchestration flowchart, tool-agent mapping, run sequence |
| Updated all `.env` references — file already exists with `DEEPSEEK_API_KEY` configured |
| Default prompt: garden planning (quick, low-token test) |
| ChatCompletion API clearly noted throughout (endpoint, model name, client setup) |
| Removed 2,000-line raw output log → replaced with runnable code + live output |